# 🧹 GIAI ĐOẠN 2: TIỀN XỬ LÝ & CHUẨN HÓA VĂN BẢN TIẾNG VIỆT (TV1)
**Thực hiện:** TV1 - Hoàng Hôn (Trưởng nhóm)

### Mục tiêu:
1. Ghép các trường văn bản (`Title`, `What I liked`, `Suggestions for improvement`).
2. Làm sạch cơ bản: chuẩn hóa Unicode NFC, xử lý emoji/emojicon, dịch teencode, sửa từ sai chính tả.
3. Trích xuất đặc trưng Lexicon cảm xúc (`pos_w`, `neg_w`, `sentiment_ratio`).
4. Tách từ tiếng Việt (`underthesea`) và loại bỏ stopwords.
5. Gán nhãn cảm xúc 3 lớp (`Positive`, `Neutral`, `Negative`) và xuất tập dữ liệu sạch.


## 1. Import các thư viện và Module Preprocessing


In [1]:
import sys, os
sys.path.append("..")
import pandas as pd
import numpy as np
from tqdm import tqdm
from src.preprocessing import TextPreprocessor

tqdm.pandas()
print("✅ Đã nạp thành công các thư viện!")


✅ Đã nạp thành công các thư viện!


## 2. Đọc dữ liệu thô từ ITviec Reviews


In [2]:
raw_path = "../data/raw/Reviews.xlsx"
df = pd.read_excel(raw_path)
print(f"Tổng số mẫu đánh giá: {len(df):,} | Số cột: {df.shape[1]}")
display(df.head(2))


Tổng số mẫu đánh giá: 8,417 | Số cột: 13


,id,Company Name,Cmt_day,Title,What I liked,Suggestions for improvement,Rating,Salary & benefits,Training & learning,Management cares about me,Culture & fun,Office & workspace,Recommend?
0,4,Accenture,March 2025,"Môi trường thoải mái, ít áp lực, có thể làm vi...","Môi trường thoải mái, ít áp lực, có thể làm vi...",Cần đánh giá KPI khách quan và phân chia công ...,4,4,4,4,4,4,Yes
1,4,Accenture,January 2025,Công ty trẻ trung năng động,"Cơ sở vật chất đẹp, công ty trẻ trung năng độn...",Nên nghiên cứu lại range lương của thì trường....,4,3,4,4,4,5,Yes


## 3. Ghép các trường văn bản


In [3]:
title = df["Title"].fillna("").astype(str)
liked = df["What I liked"].fillna("").astype(str)
suggest = df["Suggestions for improvement"].fillna("").astype(str)

df["raw_review_text"] = title + " . " + liked + " . " + suggest
print("Ví dụ văn bản thô sau khi gộp:")
print(df["raw_review_text"].iloc[0][:200] + "...")


Ví dụ văn bản thô sau khi gộp:
Môi trường thoải mái, ít áp lực, có thể làm việc hybrid, lương ở mức khá. . Môi trường thoải mái, ít áp lực, có thể làm việc hybrid, lương nên deal tốt từ lúc đầu, vì mức tăng mỗi năm ko cao. Các bene...


## 4. Khởi tạo bộ tiền xử lý và Làm sạch cơ bản (Clean Basic Text)


In [4]:
tp = TextPreprocessor(dict_dir="../data/dictionaries")

# Áp dụng làm sạch cơ bản
df["clean_basic_text"] = df["raw_review_text"].progress_apply(tp.clean_basic_text)
print("Mẫu sau khi làm sạch cơ bản:")
print(df["clean_basic_text"].iloc[0][:200] + "...")


  0%|          | 0/8417 [00:00<?, ?it/s]

  9%|▊         | 734/8417 [00:00<00:01, 7313.07it/s]

 17%|█▋        | 1466/8417 [00:00<00:00, 6964.81it/s]

 28%|██▊       | 2381/8417 [00:00<00:00, 7901.90it/s]

 40%|████      | 3369/8417 [00:00<00:00, 8665.89it/s]

...

 73%|███████▎  | 6143/8417 [00:00<00:00, 8683.00it/s]

 83%|████████▎ | 7015/8417 [00:00<00:00, 7883.82it/s]

 94%|█████████▍| 7911/8417 [00:00<00:00, 8184.86it/s]

100%|██████████| 8417/8417 [00:01<00:00, 8293.85it/s]

Mẫu sau khi làm sạch cơ bản:
môi trường thoải mái ít áp lực có thể làm việc hybrid lương ở mức khá môi trường thoải mái ít áp lực có thể làm việc hybrid lương nên thỏa thuận tốt từ lúc đầu vì mức tăng mỗi năm không cao các phúc l...


## 5. Trích xuất đặc trưng Thống kê Lexicon Cảm xúc


In [5]:
lex_feats = df["clean_basic_text"].apply(tp.calc_sentiment_features)
lex_df = pd.DataFrame(list(lex_feats))

for col in lex_df.columns:
    df[col] = lex_df[col]

display(df[["clean_basic_text", "pos_w", "neg_w", "pos_e", "neg_e", "sentiment_ratio"]].head())


,clean_basic_text,pos_w,neg_w,pos_e,neg_e,sentiment_ratio
0,môi trường thoải mái ít áp lực có thể làm việc...,0,0,0,0,0.0
1,công ty trẻ trung năng động cơ sở vật chất đẹp...,0,0,0,0,0.0
2,môi trường làm việc thoải mái vui vẻ mội người...,0,0,0,0,0.0
3,tốt environment for fresher tốt env for freshe...,0,0,0,0,0.0
4,môi trường làm việc thoải mái vui vẻ văn phòng...,1,0,0,0,1.0


## 6. Làm sạch nâng cao: Tách từ Tiếng Việt (underthesea) & Lọc Stopwords


In [6]:
df["clean_advance_text"] = df["raw_review_text"].progress_apply(
    lambda x: tp.clean_advance_text(x, remove_stopwords=True)
)

print("Mẫu sau khi tách từ và lọc stopwords:")
print(df["clean_advance_text"].iloc[0][:200] + "...")


  0%|          | 0/8417 [00:00<?, ?it/s]

  0%|          | 2/8417 [00:00<12:11, 11.51it/s]

  0%|          | 21/8417 [00:00<01:32, 90.75it/s]

  0%|          | 34/8417 [00:00<01:19, 105.65it/s]

  1%|          | 47/8417 [00:00<01:28, 94.93it/s] 

...

 99%|█████████▉| 8353/8417 [00:38<00:00, 278.46it/s]

100%|█████████▉| 8382/8417 [00:38<00:00, 262.99it/s]

100%|█████████▉| 8409/8417 [00:38<00:00, 240.12it/s]

100%|██████████| 8417/8417 [00:38<00:00, 217.99it/s]

Mẫu sau khi tách từ và lọc stopwords:
môi_trường thoải_mái áp_lực làm_việc hybrid lương môi_trường thoải_mái áp_lực làm_việc hybrid lương thỏa_thuận tốt đầu không phúc_lợi cơ_bản đầy_đủ công_ty dịch_vụ trau_dồi cải_thiện chuyên_môn ot lươ...


## 7. Gán nhãn Cảm xúc & Thống kê phân bố nhãn


In [7]:
df["sentiment"] = df["Rating"].apply(tp.map_sentiment_label)

sentiment_dist = df["sentiment"].value_counts()
sentiment_pct = df["sentiment"].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({"Số lượng": sentiment_dist, "Tỷ lệ (%)": sentiment_pct.round(2)})

print("📊 Bảng phân bố Sentiment:")
display(summary_df)


📊 Bảng phân bố Sentiment:


,Số lượng,Tỷ lệ (%)
Positive,6208,73.76
Neutral,1639,19.47
Negative,570,6.77


## 8. Lưu tập dữ liệu đã làm sạch vào `data/processed/`


In [8]:
os.makedirs("../data/processed", exist_ok=True)
out_xlsx = "../data/processed/reviews_cleaned.xlsx"
out_csv = "../data/processed/reviews_cleaned.csv"

df.to_excel(out_xlsx, index=False)
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"""🎉 Đã lưu thành công tập dữ liệu sạch tại:
1. {out_xlsx}
2. {out_csv}""")


🎉 Đã lưu thành công tập dữ liệu sạch tại:
1. ../data/processed/reviews_cleaned.xlsx
2. ../data/processed/reviews_cleaned.csv
